# Transformers: character-level language model
## Yoav Ram

We will see here the [**Transformer** architecture](https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)).
Transformers are the basis of large language models like OpenAI's [GPT](https://en.wikipedia.org/wiki/Generative_pre-trained_transformer)--the "T" stands for "Transformer".

Here, we apply transformers to the same problem we applied RNN and GRU: text generation by pretraining a character level model.

In [1]:
%matplotlib inline
import matplotlib.pyplot as plt

import jax 
import jax.numpy as np
print('jax', jax.__version__, jax.default_backend())
import optax

from collections import Counter
import pickle
import glob

jax 0.9.2 gpu


# Data

The data is just text data, in this case Shakespear's writing.

In [2]:
filename = 'data/shakespear3.txt'
with open(filename, 'rt') as f:
    text = f.read()

print("Number of characters: {}".format(len(text)))
print("Number of unique characters: {}".format(len(set(text))))
print("Number of lines: {}".format(text.count('\n')))
print("Number of words: {}".format(text.count(' ')))
print()
print("Excerpt:")
print("*" * len("Excerpt:"))
print(text[:500])

Number of characters: 4573338
Number of unique characters: 67
Number of lines: 167204
Number of words: 696192

Excerpt:
********
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


We start by creating 
- a list `chars` of the unique characters
- `data_size` the number of total characters
- `vocab_size` the number of unique characters
- `int_to_char` a dictionary from index to char
- `char_to_int` a dictionary from char to index
and then we convert `data` from a string to a NumPy array of integers representing the chars.

In [3]:
chars = sorted(set(text))
data_size, vocab_size = len(text), len(chars)

# char to int and vice versa
int_to_char = dict(enumerate(chars)) #  == { i: ch for i,ch in enumerate(chars) }
char_to_int = dict(zip(int_to_char.values(), int_to_char.keys())) # { ch: i for i,ch in enumerate(chars) }

def onehot_encode(text):
    ints = [char_to_int[c] for c in text]
    ints = np.array(ints, dtype=int)
    return jax.nn.one_hot(ints, vocab_size)

def onehot_decode(data):
    ints = data.argmax(axis=1).tolist()
    chars = (int_to_char[k] for k in ints)
    return str.join('', chars)

X = onehot_encode(text)

# Transformer model

Unlike RNN and GRU which process sequences step by step maintaining a hidden state, the Transformer processes the entire sequence at once using **self-attention** — each position can directly attend to all previous positions.

## Multi-head self-attention

For each position $i$ in the input sequence $x$, we compute **query**, **key**, and **value** vectors:
$$
q_i = W^q x_i, \quad k_j = W^k x_j, \quad v_j = W^v x_j
$$

The **attention weight** of position $i$ to position $j$ is:
$$
w_{ij} = \text{softmax}_j\left(\frac{q_i \cdot k_j}{\sqrt{d_k}}\right)
$$

The output for position $i$ is a weighted sum of values: $z_i = \sum_j w_{ij} v_j$.

A **causal mask** ensures position $i$ only attends to $j \le i$, preserving the autoregressive property (just as in RNN/GRU, character $i$ depends on $j < i$ but not $j > i$).

In **multi-head** attention, we split $q$, $k$, $v$ into multiple heads that attend independently, then concatenate their outputs. This allows the model to attend to different types of relationships simultaneously.

In [4]:
def layer_norm(x, eps=1e-6):
    mean = np.mean(x, axis=-1, keepdims=True)
    var = np.mean((x - mean) ** 2, axis=-1, keepdims=True)
    return (x - mean) / np.sqrt(var + eps)

def multi_head_attention(x, W_q, W_k, W_v, W_o, n_heads):
    seq_len, d_model = x.shape
    d_head = d_model // n_heads
    
    Q = x @ W_q  # (seq_len, d_model)
    K = x @ W_k
    V = x @ W_v
    
    # reshape to (n_heads, seq_len, d_head)
    Q = Q.reshape(seq_len, n_heads, d_head).transpose(1, 0, 2)
    K = K.reshape(seq_len, n_heads, d_head).transpose(1, 0, 2)
    V = V.reshape(seq_len, n_heads, d_head).transpose(1, 0, 2)
    
    # attention weights
    w_logits = Q @ K.transpose(0, 2, 1) / np.sqrt(d_head)  # (n_heads, seq_len, seq_len)
    # causal mask
    mask = np.tril(np.ones((seq_len, seq_len)))
    w_logits = w_logits - 1e10 * (1 - mask)
    w = jax.nn.softmax(w_logits, axis=-1)
    
    # weighted sum of values
    z = w @ V  # (n_heads, seq_len, d_head)
    # concatenate heads
    z = z.transpose(1, 0, 2).reshape(seq_len, d_model)
    
    return z @ W_o

## Transformer block and model

Each transformer block applies:
1. Layer normalization → multi-head self-attention → residual connection
2. Layer normalization → feed-forward network → residual connection

This is the "pre-norm" architecture, which is more stable to train than the original "post-norm" from the [Attention Is All You Need](https://arxiv.org/abs/1706.03762) paper.

The full model stacks multiple transformer blocks. Unlike RNN and GRU where the input weight matrix $W_x^h$ implicitly serves as an embedding (since $W_x^h \cdot \text{onehot}(c)$ simply selects column $c$ of the matrix), the transformer has no per-step input weight matrix — it applies the same attention mechanism to all positions. It therefore needs an explicit **learned token embedding** to convert characters to continuous vectors. Similarly, it uses a **learned positional embedding** to encode position information, since the attention mechanism is permutation-invariant and has no inherent notion of order.

In [5]:
def transformer_block(x, layer_params, n_heads):
    # self-attention with pre-norm and residual
    sa = multi_head_attention(
        layer_norm(x), 
        layer_params['W_q'], layer_params['W_k'], 
        layer_params['W_v'], layer_params['W_o'], 
        n_heads
    )
    x = x + sa
    # feed-forward with pre-norm and residual
    ff_in = layer_norm(x)
    hidden = jax.nn.relu(ff_in @ layer_params['W1'] + layer_params['b1'])
    ff = hidden @ layer_params['W2'] + layer_params['b2']
    x = x + ff
    return x

def transformer_model(params, x, n_heads):
    # convert from one-hot to integer indices
    indices = x.argmax(axis=1)
    seq_len = indices.shape[0]
    # token embedding + positional embedding
    x = params['token_emb'][indices] + params['pos_emb'][:seq_len]
    # transformer blocks
    for layer_params in params['layers']:
        x = transformer_block(x, layer_params, n_heads)
    # final layer norm and projection to logits
    x = layer_norm(x)
    logits = x @ params['W_out'] + params['b_out']
    return logits

We initialize the network parameters so we can test the model.

In [6]:
def init_layer_params(key, d_model, d_ff):
    keys = jax.random.split(key, 6)
    return {
        'W_q': jax.random.normal(keys[0], (d_model, d_model)) * 0.02,
        'W_k': jax.random.normal(keys[1], (d_model, d_model)) * 0.02,
        'W_v': jax.random.normal(keys[2], (d_model, d_model)) * 0.02,
        'W_o': jax.random.normal(keys[3], (d_model, d_model)) * 0.02,
        'W1': jax.random.normal(keys[4], (d_model, d_ff)) * 0.02,
        'b1': np.zeros((d_ff,)),
        'W2': jax.random.normal(keys[5], (d_ff, d_model)) * 0.02,
        'b2': np.zeros((d_model,)),
    }

def init_params(key, vocab_size, max_seq_len, d_model, d_ff, n_layers):
    keys = jax.random.split(key, n_layers + 3)
    return {
        'token_emb': jax.random.normal(keys[0], (vocab_size, d_model)) * 0.02,
        'pos_emb': jax.random.normal(keys[1], (max_seq_len, d_model)) * 0.02,
        'layers': [init_layer_params(keys[2 + i], d_model, d_ff) for i in range(n_layers)],
        'W_out': jax.random.normal(keys[-1], (d_model, vocab_size)) * 0.02,
        'b_out': np.zeros((vocab_size,)),
    }

## Loss function

The loss function is categorical cross-entropy (negative log-likelihood).
We use `log_softmax` for numerical stability.

In [7]:
seq_length = 50
max_seq_len = 512
d_model = 128
d_ff = 512
n_heads = 4
n_layers = 5

key = jax.random.key(0)
params = init_params(key, vocab_size, max_seq_len, d_model, d_ff, n_layers)

x, y = X[:seq_length], X[1:seq_length + 1]
logits = transformer_model(params, x, n_heads)
assert logits.shape == (seq_length, vocab_size)
print("logits shape:", logits.shape)

E0405 10:50:03.753110  565088 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[128,50]{0,1} fusion(a.1, b.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 10:50:03.753194  565088 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_1 = f32[128,128]{1,0} parameter(1)
  parameter_0 = f32[50,128]{1,0} parameter(0)
  ROOT dot = f32[128,50]{0,1} dot(parameter_1, parameter_0), lhs_contracting_dims={0}, rhs_contracting_dims={1}, backend_config={"sizes":["32"]}
}
E0405 10:50:03.775195  565091 xtile_compiler.cc:399] Fusi

E0405 10:50:06.872435  565163 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[512,50]{0,1} fusion(a.1, b.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 10:50:06.872526  565163 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_1 = f32[128,512]{1,0} parameter(1)
  parameter_0 = f32[50,128]{1,0} parameter(0)
  ROOT dot = f32[512,50]{0,1} dot(parameter_1, parameter_0), lhs_contracting_dims={0}, rhs_contracting_dims={1}, backend_config={"sizes":["32"]}
}
E0405 10:50:06.884809  565139 xtile_compiler.cc:399] Fusi

E0405 10:50:08.309251  565130 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[67,50]{0,1} fusion(a.1, b.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 10:50:08.309336  565130 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_1 = f32[128,67]{1,0} parameter(1)
  parameter_0 = f32[50,128]{1,0} parameter(0)
  ROOT dot = f32[67,50]{0,1} dot(parameter_1, parameter_0), lhs_contracting_dims={0}, rhs_contracting_dims={1}, backend_config={"sizes":["32"]}
}
E0405 10:50:08.345678  565157 xtile_compiler.cc:399] Fusion:

logits shape: (50, 67)


The NLL (negative log-likelihood, cross-entropy) for a single step is
$$
NLL = -\sum_{k=0}^{V}{x_{t+1,k} \log{\hat{y}_{t,k}}}
$$
where $k$ runs over all $V$ characters.
The total NLL is computed by averaging over all positions $t$.

In [8]:
def NLL(params, x, y):
    logits = transformer_model(params, x, n_heads)
    log_probs = jax.nn.log_softmax(logits)
    loss = -(y * log_probs).sum() / x.shape[0]
    return loss

loss = NLL(params, x, y)
print(loss)

4.202741


## Automatic differentiation with JAX

We use JAX's automatic differentiation to compute gradients. [`jax.value_and_grad`](https://jax.readthedocs.io/en/latest/jax-101/01-jax-basics.html#value-and-grad) returns both the loss value and the gradient of the loss with respect to the parameters.

In [9]:
backprop = jax.value_and_grad(NLL)

loss, grads = backprop(params, x, y)
for k in params:
    if k == 'layers':
        for layer_params, layer_grads in zip(params[k], grads[k]):
            for p_name in layer_params:
                assert layer_params[p_name].shape == layer_grads[p_name].shape
    else:
        assert params[k].shape == grads[k].shape

E0405 10:50:13.589396  565097 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[67,128]{1,0} fusion(args_0_.1, args_1_.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 10:50:13.589499  565097 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_0 = f32[50,67]{1,0} parameter(0)
  parameter_1 = f32[50,128]{1,0} parameter(1)
  ROOT dot_general.0 = f32[67,128]{1,0} dot(parameter_0, parameter_1), lhs_contracting_dims={0}, rhs_contracting_dims={0}, backend_config={"sizes":["32"]}
}
E0405 10:50:13.615907  565094 xtile_c

E0405 10:50:14.276173  565102 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[128,50]{0,1} fusion(args_0_.1, args_1_.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"4","output_tiles":[{"sizes":["64","64"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 10:50:14.276263  565102 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_1 = f32[128,67]{1,0} parameter(1)
  parameter_0 = f32[50,67]{1,0} parameter(0)
  ROOT dot = f32[128,50]{0,1} dot(parameter_1, parameter_0), lhs_contracting_dims={1}, rhs_contracting_dims={1}, backend_config={"sizes":["64"]}
}
E0405 10:50:14.320617  565072 xtile_compiler.cc:3

E0405 10:50:15.667950  565128 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[128,512]{1,0} fusion(args_0_.1, args_1_.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"4","output_tiles":[{"sizes":["64","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 10:50:15.668076  565128 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_0 = f32[50,128]{1,0} parameter(0)
  parameter_1 = f32[50,512]{1,0} parameter(1)
  ROOT dot_general.0 = f32[128,512]{1,0} dot(parameter_0, parameter_1), lhs_contracting_dims={0}, rhs_contracting_dims={0}, backend_config={"sizes":["32"]}
}
E0405 10:50:15.694013  565116 xtile

E0405 10:50:16.309048  565072 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[512,50]{0,1} fusion(args_0_.1, args_1_.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 10:50:16.309123  565072 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_1 = f32[512,128]{1,0} parameter(1)
  parameter_0 = f32[50,128]{1,0} parameter(0)
  ROOT dot = f32[512,50]{0,1} dot(parameter_1, parameter_0), lhs_contracting_dims={1}, rhs_contracting_dims={1}, backend_config={"sizes":["32"]}
}
E0405 10:50:16.330433  565125 xtile_compiler.

E0405 10:50:17.324673  565118 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[512,128]{1,0} fusion(args_0_.1, args_1_.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"4","output_tiles":[{"sizes":["64","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 10:50:17.324795  565118 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_0 = f32[50,512]{1,0} parameter(0)
  parameter_1 = f32[50,128]{1,0} parameter(1)
  ROOT dot_general.0 = f32[512,128]{1,0} dot(parameter_0, parameter_1), lhs_contracting_dims={0}, rhs_contracting_dims={0}, backend_config={"sizes":["32"]}
}
E0405 10:50:17.327757  565124 xtile

E0405 10:50:18.500633  565101 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[128,128]{1,0} fusion(args_0_.1, args_1_.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"4","output_tiles":[{"sizes":["32","256"]}],"num_ctas":1,"num_stages":3,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 10:50:18.500734  565101 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_0 = f32[50,128]{1,0} parameter(0)
  parameter_1 = f32[50,128]{1,0} parameter(1)
  ROOT dot_general.0 = f32[128,128]{1,0} dot(parameter_0, parameter_1), lhs_contracting_dims={0}, rhs_contracting_dims={0}, backend_config={"sizes":["32"]}
}
E0405 10:50:18.527625  565117 xtile

E0405 10:50:19.255315  565100 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[128,50]{0,1} fusion(args_0_.1, args_1_.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"4","output_tiles":[{"sizes":["64","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 10:50:19.255402  565100 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_1 = f32[128,128]{1,0} parameter(1)
  parameter_0 = f32[50,128]{1,0} parameter(0)
  ROOT dot = f32[128,50]{0,1} dot(parameter_1, parameter_0), lhs_contracting_dims={1}, rhs_contracting_dims={1}, backend_config={"sizes":["32"]}
}
E0405 10:50:19.258608  565103 xtile_compiler.c

# Adam optimizer with Optax

We can use a JAX implementation of the Adam optimizer from the [Optax](https://optax.readthedocs.io/) library.
We first create the optimizer and initialize its state.

In [10]:
optimizer = optax.adam(learning_rate=0.001) # 0.001 is the default from Kingma et al 2014
opt_state = optimizer.init(params)

We then use the optimizer to compute the updates, and apply them.

In [11]:
loss, grads = backprop(params, x, y)
updates, opt_state = optimizer.update(grads, opt_state, params)
params = optax.apply_updates(params, updates)

# JITing the training step

We write a function that does all this, and pass it to `jax.jit`, which [just-in-time compiles the function](https://jax.readthedocs.io/en/latest/jax-101/02-jitting.html) so it can be executed efficiently in XLA.

In [12]:
@jax.jit
def update_params(params, opt_state, x, y):
    loss, grads = backprop(params, x, y)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

In [13]:
%timeit update_params(params, opt_state, x, y)

E0405 10:50:27.437915  565125 xtile_compiler.cc:399] Fusion: gemm_fusion_dot = f32[128,67]{1,0} fusion(div.548, add_any.0), kind=kCustom, calls=gemm_fusion_dot_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"4","output_tiles":[{"sizes":["32","256"]}],"num_ctas":1,"num_stages":3,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 10:50:27.438044  565125 xtile_compiler.cc:401] Computation: gemm_fusion_dot_computation.clone {
  parameter_0.31 = f32[50,128]{1,0} parameter(0)
  parameter_1.31 = f32[50,67]{1,0} parameter(1)
  ROOT dot.86 = f32[128,67]{1,0} dot(parameter_0.31, parameter_1.31), lhs_contracting_dims={0}, rhs_contracting_dims={0}, backend_config={"sizes":["32"]}
}
E0405 10:50:27.447397  565116 xtile_compiler.cc:399] Fusion: gemm

E0405 10:50:27.657053  565163 xtile_compiler.cc:399] Fusion: gemm_fusion_dot.34 = f32[384,128]{0,1} fusion(div.438, bitcast.383, bitcast.386, bitcast.389), kind=kCustom, calls=gemm_fusion_dot.34_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"4","output_tiles":[{"sizes":["32","256"]}],"num_ctas":1,"num_stages":3,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 10:50:27.657225  565163 xtile_compiler.cc:401] Computation: gemm_fusion_dot.34_computation.clone {
  parameter_1.82 = f32[50,128]{1,0} parameter(1)
  parameter_2.11 = f32[50,128]{1,0} parameter(2)
  parameter_3.5 = f32[50,128]{1,0} parameter(3)
  concatenate.35 = f32[50,384]{1,0} concatenate(parameter_1.82, parameter_2.11, parameter_3.5), dimensions={1}
  parameter_0.82 = f32[

543 μs ± 126 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [14]:
params, opt_state, loss = update_params(params, opt_state, x, y)
print(loss)
params, opt_state, loss = update_params(params, opt_state, x, y)
print(loss)

3.6606255
3.4057035


# Sampling from the network

Instead of a `predict` function, we have a `sample` function, which, given the parameters and the number of samples we want, produces a sample of text from the network.

It does so by drawing a random seed for $x_0$ and drawing $x_t$ for $t>0$ from the distribution given by $\widehat y_t$.

Unlike RNN/GRU which maintain a hidden state, the transformer must re-process the entire sequence at each step to generate the next character.

In [15]:
def sample(params, num_samples, key):
    x = np.zeros((num_samples, vocab_size), dtype=float)
    key, subkey = jax.random.split(key)
    seed_char = jax.random.choice(subkey, vocab_size)
    x = x.at[0, seed_char].set(1)
    
    for t in range(1, num_samples):
        logits = transformer_model(params, x[:t], n_heads)[t-1]
        yhat = jax.nn.softmax(logits)
        # draw from output distribution
        key, subkey = jax.random.split(key)
        i = jax.random.choice(subkey, vocab_size, p=yhat)
        x = x.at[t, i].set(1)
    return onehot_decode(x)

print(sample(params, 100, jax.random.key(1)))

E0405 10:51:13.592255  565075 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[67,9]{0,1} fusion(a.1, b.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"4","output_tiles":[{"sizes":["64","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 10:51:13.592348  565075 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_1 = f32[128,67]{1,0} parameter(1)
  parameter_0 = f32[9,128]{1,0} parameter(0)
  ROOT dot = f32[67,9]{0,1} dot(parameter_1, parameter_0), lhs_contracting_dims={0}, rhs_contracting_dims={1}, backend_config={"sizes":["32"]}
}
E0405 10:51:13.617323  565076 xtile_compiler.cc:399] Fusion: gem

E0405 10:51:18.374230  565120 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[67,10]{0,1} fusion(a.1, b.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"4","output_tiles":[{"sizes":["64","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 10:51:18.374312  565120 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_1 = f32[128,67]{1,0} parameter(1)
  parameter_0 = f32[10,128]{1,0} parameter(0)
  ROOT dot = f32[67,10]{0,1} dot(parameter_1, parameter_0), lhs_contracting_dims={0}, rhs_contracting_dims={1}, backend_config={"sizes":["32"]}
}
E0405 10:51:18.385287  565106 xtile_compiler.cc:399] Fusion: 

E0405 10:51:22.736260  565148 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[67,11]{0,1} fusion(a.1, b.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"4","output_tiles":[{"sizes":["64","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 10:51:22.736377  565148 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_1 = f32[128,67]{1,0} parameter(1)
  parameter_0 = f32[11,128]{1,0} parameter(0)
  ROOT dot = f32[67,11]{0,1} dot(parameter_1, parameter_0), lhs_contracting_dims={0}, rhs_contracting_dims={1}, backend_config={"sizes":["32"]}
}
E0405 10:51:22.761816  565157 xtile_compiler.cc:399] Fusion: 

E0405 10:51:26.431965  565149 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[67,12]{0,1} fusion(a.1, b.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 10:51:26.432058  565149 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_1 = f32[128,67]{1,0} parameter(1)
  parameter_0 = f32[12,128]{1,0} parameter(0)
  ROOT dot = f32[67,12]{0,1} dot(parameter_1, parameter_0), lhs_contracting_dims={0}, rhs_contracting_dims={1}, backend_config={"sizes":["32"]}
}
E0405 10:51:26.436513  565070 xtile_compiler.cc:399] Fusion:

E0405 10:51:30.955853  565132 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[67,13]{0,1} fusion(a.1, b.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"4","output_tiles":[{"sizes":["64","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 10:51:30.955948  565132 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_1 = f32[128,67]{1,0} parameter(1)
  parameter_0 = f32[13,128]{1,0} parameter(0)
  ROOT dot = f32[67,13]{0,1} dot(parameter_1, parameter_0), lhs_contracting_dims={0}, rhs_contracting_dims={1}, backend_config={"sizes":["32"]}
}
E0405 10:51:30.984211  565114 xtile_compiler.cc:399] Fusion: 

...

E0405 10:59:21.694324  565083 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[98,67]{1,0} fusion(a.1, b.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 10:59:21.694418  565083 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_0 = f32[98,128]{1,0} parameter(0)
  parameter_1 = f32[128,67]{1,0} parameter(1)
  ROOT dot_general.0 = f32[98,67]{1,0} dot(parameter_0, parameter_1), lhs_contracting_dims={1}, rhs_contracting_dims={0}, backend_config={"sizes":["32"]}
}
E0405 10:59:21.697691  565071 xtile_compiler.cc:39

E0405 10:59:23.935059  565128 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[99,128]{1,0} fusion(a.1, b.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 10:59:23.935140  565128 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_0 = f32[99,128]{1,0} parameter(0)
  parameter_1 = f32[128,128]{1,0} parameter(1)
  ROOT dot_general.0 = f32[99,128]{1,0} dot(parameter_0, parameter_1), lhs_contracting_dims={1}, rhs_contracting_dims={0}, backend_config={"sizes":["32"]}
}
E0405 10:59:23.948494  565146 xtile_compiler.cc

E0405 10:59:27.068040  565130 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[99,512]{1,0} fusion(a.1, b.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 10:59:27.068125  565130 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_0 = f32[99,128]{1,0} parameter(0)
  parameter_1 = f32[128,512]{1,0} parameter(1)
  ROOT dot_general.0 = f32[99,512]{1,0} dot(parameter_0, parameter_1), lhs_contracting_dims={1}, rhs_contracting_dims={0}, backend_config={"sizes":["32"]}
}
E0405 10:59:27.072974  565155 xtile_compiler.cc

E0405 10:59:28.059416  565083 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[99,67]{1,0} fusion(a.1, b.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 10:59:28.059511  565083 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_0 = f32[99,128]{1,0} parameter(0)
  parameter_1 = f32[128,67]{1,0} parameter(1)
  ROOT dot_general.0 = f32[99,67]{1,0} dot(parameter_0, parameter_1), lhs_contracting_dims={1}, rhs_contracting_dims={0}, backend_config={"sizes":["32"]}
}
E0405 10:59:28.099926  565147 xtile_compiler.cc:39

G;VBwnnL,eOrrsjXA &h B- Ak,azMzyguh.rckeaLdeenG-MvMyNrBrcdzAfgmeGSFlY
naOMTeGlqV-nuCTF,SGiCFth uheNa


# Training the network

We setup the training - the sequence length, the number of batches, parameter initialization, Adam optimizer.

In [16]:
seq_length = 50
max_seq_len = 512
max_batches = 10000000
pos = 0
batch = 0
losses = []
key = jax.random.key(86)

d_model = 128
d_ff = 512
n_heads = 4
n_layers = 5

params = init_params(key, vocab_size, max_seq_len, d_model, d_ff, n_layers)

backprop = jax.value_and_grad(NLL)

# Warmup + cosine decay schedule over the full training run.
warmup_steps = max(1000, max_batches // 100)
schedule = optax.warmup_cosine_decay_schedule(
    init_value=1e-4,
    peak_value=1e-3,
    warmup_steps=warmup_steps,
    decay_steps=max_batches,
)
optimizer = optax.chain(
    optax.clip_by_global_norm(5.0),
    optax.adam(learning_rate=schedule),
)
opt_state = optimizer.init(params)

# Simple early stopping based on batch loss.
early_stop_check_every = 50000
early_stop_patience = 50
early_stop_min_delta = 0.005
best_loss = np.inf
bad_count = 0

Now we can train the Transformer!

In [17]:
%%time
while batch <= max_batches:
    if pos + seq_length + 1 >= data_size:
        pos = 0

    x = X[pos : pos + seq_length]
    y = X[pos + 1 : pos + seq_length + 1]
    pos += seq_length

    params, opt_state, loss = update_params(params, opt_state, x, y)
    losses.append(loss)

    if batch % (max_batches // 10) == 0:
        print('batch {:d}, loss {:.6f}, pos {}'.format(batch, loss, pos))
        print()

        with open("data/transformer-jax-params-{}.pkl".format(batch), 'wb') as file:
            pickle.dump(params, file)

        key, subkey = jax.random.split(key)
        sample_text = sample(params, 200, subkey)
        print(sample_text)
        print('-'*80)

    if batch % early_stop_check_every == 0:
        if loss < best_loss - early_stop_min_delta:
            best_loss = loss
            bad_count = 0
        else:
            bad_count += 1
            if bad_count >= early_stop_patience:
                print('early stopping at batch {:d}, loss {:.6f}'.format(batch, loss))
                break

    batch += 1

batch 0, loss 4.258337, pos 50



E0405 10:59:46.159982  565106 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[100,128]{1,0} fusion(a.1, b.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 10:59:46.160067  565106 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_0 = f32[100,128]{1,0} parameter(0)
  parameter_1 = f32[128,128]{1,0} parameter(1)
  ROOT dot_general.0 = f32[100,128]{1,0} dot(parameter_0, parameter_1), lhs_contracting_dims={1}, rhs_contracting_dims={0}, backend_config={"sizes":["32"]}
}
E0405 10:59:46.203555  565135 xtile_compiler

E0405 10:59:49.177500  565074 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[100,512]{1,0} fusion(a.1, b.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 10:59:49.177640  565074 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_0 = f32[100,128]{1,0} parameter(0)
  parameter_1 = f32[128,512]{1,0} parameter(1)
  ROOT dot_general.0 = f32[100,512]{1,0} dot(parameter_0, parameter_1), lhs_contracting_dims={1}, rhs_contracting_dims={0}, backend_config={"sizes":["64"]}
}
E0405 10:59:49.198407  565119 xtile_compiler

E0405 10:59:50.385616  565130 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[100,67]{1,0} fusion(a.1, b.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"4","output_tiles":[{"sizes":["64","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 10:59:50.385713  565130 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_0 = f32[100,128]{1,0} parameter(0)
  parameter_1 = f32[128,67]{1,0} parameter(1)
  ROOT dot_general.0 = f32[100,67]{1,0} dot(parameter_0, parameter_1), lhs_contracting_dims={1}, rhs_contracting_dims={0}, backend_config={"sizes":["32"]}
}
E0405 10:59:50.420729  565145 xtile_compiler.cc:

E0405 10:59:52.579926  565081 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[101,128]{1,0} fusion(a.1, b.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 10:59:52.580015  565081 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_0 = f32[101,128]{1,0} parameter(0)
  parameter_1 = f32[128,128]{1,0} parameter(1)
  ROOT dot_general.0 = f32[101,128]{1,0} dot(parameter_0, parameter_1), lhs_contracting_dims={1}, rhs_contracting_dims={0}, backend_config={"sizes":["32"]}
}
E0405 10:59:52.584771  565136 xtile_compiler

...

batch 3000000, loss 0.858182, pos 3654450



LEANS:
'Whill not you speak excelled
Will cover! bry ten Ly net; s I inaso's, t ad, Oporiged ashe thor ly hon I atole win'sth s, Nthes s I ars why inages head thesteagone Hor y--wigopes, gonat hes age
--------------------------------------------------------------------------------


batch 4000000, loss 1.532672, pos 3348150



NCELLUS:
O, man, i' faith.

EARL OF SOUGHBY:
But, wid, Ishorneaiddey nearneake meakelddd trnesididgiddd Aldddd!
Barsid tobrne molld!
IO who mid hid, th.
Barshomo widdshalcrgorgrnear ccarnearshar thar 
--------------------------------------------------------------------------------


early stopping at batch 4800000, loss 1.386272
CPU times: user 2h 58min 35s, sys: 28min 53s, total: 3h 27min 28s
Wall time: 2h 51min 25s


In [18]:
with open("data/transformer-jax-params-{}.pkl".format(batch), 'wb') as file:
    pickle.dump(params, file)

# Load parameters from specific batch

In [19]:
paths = sorted(glob.glob("data/transformer-jax-params-*.pkl"))
if not paths:
    raise FileNotFoundError("No Transformer checkpoint files found in data/")

latest = paths[-1]
print("Loading checkpoint:", latest)
with open(latest, 'rb') as file:
    params = pickle.load(file)

Loading checkpoint: data/transformer-jax-params-4800000.pkl


In [20]:
print(sample(params, 500, jax.random.key(12)))

E0405 14:02:49.183143  565098 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[200,128]{1,0} fusion(a.1, b.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 14:02:49.183234  565098 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_0 = f32[200,128]{1,0} parameter(0)
  parameter_1 = f32[128,128]{1,0} parameter(1)
  ROOT dot_general.0 = f32[200,128]{1,0} dot(parameter_0, parameter_1), lhs_contracting_dims={1}, rhs_contracting_dims={0}, backend_config={"sizes":["32"]}
}
E0405 14:02:49.225641  565125 xtile_compiler

E0405 14:02:53.232949  565105 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[200,67]{1,0} fusion(a.1, b.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"4","output_tiles":[{"sizes":["64","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 14:02:53.233046  565105 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_0 = f32[200,128]{1,0} parameter(0)
  parameter_1 = f32[128,67]{1,0} parameter(1)
  ROOT dot_general.0 = f32[200,67]{1,0} dot(parameter_0, parameter_1), lhs_contracting_dims={1}, rhs_contracting_dims={0}, backend_config={"sizes":["32"]}
}
E0405 14:02:53.254983  565131 xtile_compiler.cc:

E0405 14:02:55.528380  565069 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[201,128]{1,0} fusion(a.1, b.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 14:02:55.528482  565069 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_0 = f32[201,128]{1,0} parameter(0)
  parameter_1 = f32[128,128]{1,0} parameter(1)
  ROOT dot_general.0 = f32[201,128]{1,0} dot(parameter_0, parameter_1), lhs_contracting_dims={1}, rhs_contracting_dims={0}, backend_config={"sizes":["64"]}
}
E0405 14:02:55.558640  565121 xtile_compiler

E0405 14:02:59.954753  565125 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[201,67]{1,0} fusion(a.1, b.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"4","output_tiles":[{"sizes":["64","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 14:02:59.954841  565125 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_0 = f32[201,128]{1,0} parameter(0)
  parameter_1 = f32[128,67]{1,0} parameter(1)
  ROOT dot_general.0 = f32[201,67]{1,0} dot(parameter_0, parameter_1), lhs_contracting_dims={1}, rhs_contracting_dims={0}, backend_config={"sizes":["32"]}
}
E0405 14:02:59.993351  565143 xtile_compiler.cc:

E0405 14:03:02.259079  565117 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[202,128]{1,0} fusion(a.1, b.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 14:03:02.259152  565117 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_0 = f32[202,128]{1,0} parameter(0)
  parameter_1 = f32[128,128]{1,0} parameter(1)
  ROOT dot_general.0 = f32[202,128]{1,0} dot(parameter_0, parameter_1), lhs_contracting_dims={1}, rhs_contracting_dims={0}, backend_config={"sizes":["32"]}
}
E0405 14:03:02.269511  565156 xtile_compiler

...

E0405 14:39:01.431726  565116 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[498,128]{1,0} fusion(a.1, b.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 14:39:01.431812  565116 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_0 = f32[498,128]{1,0} parameter(0)
  parameter_1 = f32[128,128]{1,0} parameter(1)
  ROOT dot_general.0 = f32[498,128]{1,0} dot(parameter_0, parameter_1), lhs_contracting_dims={1}, rhs_contracting_dims={0}, backend_config={"sizes":["32"]}
}
E0405 14:39:01.438146  565069 xtile_compiler

E0405 14:39:05.992779  565091 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[498,67]{1,0} fusion(a.1, b.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 14:39:05.992876  565091 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_0 = f32[498,128]{1,0} parameter(0)
  parameter_1 = f32[128,67]{1,0} parameter(1)
  ROOT dot_general.0 = f32[498,67]{1,0} dot(parameter_0, parameter_1), lhs_contracting_dims={1}, rhs_contracting_dims={0}, backend_config={"sizes":["32"]}
}
E0405 14:39:06.010680  565131 xtile_compiler.cc

E0405 14:39:08.419679  565105 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[499,128]{1,0} fusion(a.1, b.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 14:39:08.419760  565105 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_0 = f32[499,128]{1,0} parameter(0)
  parameter_1 = f32[128,128]{1,0} parameter(1)
  ROOT dot_general.0 = f32[499,128]{1,0} dot(parameter_0, parameter_1), lhs_contracting_dims={1}, rhs_contracting_dims={0}, backend_config={"sizes":["32"]}
}
E0405 14:39:08.422571  565153 xtile_compiler

E0405 14:39:13.216369  565153 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[499,67]{1,0} fusion(a.1, b.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0405 14:39:13.216447  565153 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_0 = f32[499,128]{1,0} parameter(0)
  parameter_1 = f32[128,67]{1,0} parameter(1)
  ROOT dot_general.0 = f32[499,67]{1,0} dot(parameter_0, parameter_1), lhs_contracting_dims={1}, rhs_contracting_dims={0}, backend_config={"sizes":["32"]}
}
E0405 14:39:13.254539  565138 xtile_compiler.cc

Droop; a' the immortal tears.

Servant:
We must comere the re mochino o wnalllelthindo hakntocumelto e? isure tonele o---deltheltinderumeltochy: akndo ithelakninellthale cutochinaknvaltheltocufo merurelthakneltocusale? pelto CI inakne pultocheltoretalthalthinonakno wakneleltheltrelthaknde, theshaknthaknalto shaltontheltheaknalto ithisongrelthelthakndo ochaknino prelalthelthelthakndelelthakinalthaseaknelto reelto urelthealthinelthaso thelthindelelthechinelthelthindo thalthaknelenalthelthelalthelt


# References

- [Vaswani et al. 2017](http://arxiv.org/abs/1706.03762): _Attention Is All You Need_, the fundamental paper on transformers.


# Colophon
This notebook was written by [Yoav Ram](http://python.yoavram.com).

This work is licensed under a [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/) International License.

![Python logo](https://www.python.org/static/community_logos/python-logo.png)